In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import contextily as cx

import geopandas as gpd
from sklearn.cluster import DBSCAN
import folium
import seaborn as sns
from pysal.viz import mapclassify


In [13]:
df = pd.read_csv('../data/movement.csv')

cols = ['start_lat', 'start_lon', 'end_lat', 'end_lon']
df = df[cols].dropna().reset_index(drop=True)


df.head()

,start_lat,start_lon,end_lat,end_lon
0,55.658398,12.514628,55.658348,12.515684
1,55.658348,12.515684,55.659286,12.519309
2,55.659286,12.519309,55.677685,12.522237
3,55.677685,12.522237,55.676945,12.520396
4,55.676945,12.520396,55.655346,12.537441


In [14]:
start_pts = gpd.GeoDataFrame(


    geometry=gpd.points_from_xy(df['start_lon'], df['start_lat']),
    crs='EPSG:4326'
).to_crs(epsg=25832)

df['start_X'] =   start_pts.geometry.x
df['start_Y']      = start_pts.geometry.y
#print(df['start_X'])



end_pts = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(df['end_lon'], df['end_lat']),
    crs='EPSG:4326'
).to_crs(epsg=25832)

df['end_X'] =end_pts.geometry.x

df['end_Y'] = end_pts.geometry.y
print('metres')


df[['start_X', 'start_Y', 'end_X', 'end_Y']].head()

metres


,start_X,start_Y,end_X,end_Y
0,721078.781674,6.173663e+06,721145.458425,6.173661e+06
1,721145.458425,6.173661e+06,721368.078357,6.173777e+06
2,721368.078357,6.173777e+06,721448.159371,6.175832e+06
3,721448.159371,6.175832e+06,721336.647596,6.175744e+06
4,721336.647596,6.175744e+06,722530.439044,6.173396e+06


In [15]:
EPS         = 300
MIN_SAMPLES = 100

algo  = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)

algo.fit(df[['start_X', 'start_Y']])
df['start_cluster'] = pd.Series(algo.labels_, index= df.index)

algo =    DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)

algo.fit(df[['end_X', 'end_Y']])
df['end_cluster']  = pd.Series(algo.labels_, index=df.index)

n_start = df[df['start_cluster']  != -1]['start_cluster'].nunique()
#print(f'Srat c: {n_start}')

n_end =   df[df['end_cluster']!= -1]['end_cluster'].nunique()


#print(f'End c : {n_end}')

In [18]:


MIN_TRIPS = 100

both=  df[(df['start_cluster'] != -1) & (df['end_cluster'] != -1)]

lat_lon = both.groupby(['start_cluster', 'end_cluster'])[['start_lat', 'start_lon', 'end_lat', 'end_lon']].mean()

lat_lon['trip_count'] = both.groupby(['start_cluster', 'end_cluster']).size()
lat_lon =  lat_lon.reset_index()

lat_lon  =    lat_lon[lat_lon['trip_count'] >= MIN_TRIPS]

lat_lon=  lat_lon.sort_values('trip_count', ascending=False)


lat_lon = lat_lon.reset_index(drop=True)

lat_lon = lat_lon[
    lat_lon['start_lat'].between(55.5, 55.85) &
    lat_lon['start_lon'].between(12.2, 12.8)
]

print(f'Corridors with >= {MIN_TRIPS} trips: {len(lat_lon)}')

lat_lon.head(7)

Corridors with >= 100 trips: 88


,start_cluster,end_cluster,start_lat,start_lon,end_lat,end_lon,trip_count
0,0,0,55.698934,12.549017,55.699017,12.548899,323399
2,0,1,55.705498,12.543334,55.630321,12.648716,8594
3,1,0,55.630301,12.648720,55.702952,12.547384,8587
4,2,0,55.785532,12.522597,55.709666,12.543702,8366
5,0,2,55.710128,12.543883,55.785546,12.522728,8136
6,0,3,55.675404,12.557269,55.628265,12.576100,7156
7,3,0,55.628099,12.576082,55.675187,12.557326,7137


In [21]:
AIRPORTS = [
    ((55.630397, 12.648908), 0.005),
]

m = folium.Map(

    
    location=[55.676, 12.568],
    zoom_start=11,
    tiles='OpenStreetMap'
)

for idx, row in lat_lon.iterrows():


    count = int(row['trip_count'])
    radius = 15

    is_airport = False
    for ap, margin in AIRPORTS:


        if abs(row['start_lat'] - ap[0]) < margin and abs(row['start_lon'] - ap[1]) < margin:
            is_airport = True

    color = 'blue' if is_airport else 'green'

    popup = (
        f"Trips: {count}<br>"
        
        f"Origin: ({row['start_lat']:.4f}, {row['start_lon']:.4f})<br>"
        f"Dest  : ({row['end_lat']:.4f}, {row['end_lon']:.4f})"
    )

    folium.CircleMarker(

        location=[float(row['start_lat']), float(row['start_lon'])],
        radius=radius,
        
        color=color,
        weight=0.8,
        fill=True,

        fill_opacity=0.5,
        popup=folium.Popup(popup, max_width=200)
    ).add_to(m)

    folium.PolyLine(
       
        locations=[
            [float(row['start_lat']), float(row['start_lon'])],
            
            [float(row['end_lat']),   float(row['end_lon'])]    
        ],
        color=color,
        
        weight=1,
        opacity=0.6
    ).add_to(m)

m.save('DBSCAN_clustering_map.html')